# Banka Kampanyası Verisi: Kim Vadeli Mevduat Açar?

[![Colab'da Aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haticeozbolat01/veri-projeleri/blob/main/02-banka-kampanyasi-vadeli-mevduat.ipynb)

Bu not defteri, haticeozbolat.com'daki **Banka Kampanyası Verisi** yazısının kodlarını içeriyor. Yazı: [Yazıyı oku]([YAZI LİNKİ])

**Veri kaynağı:** [UCI Machine Learning Repository, Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), CC BY 4.0. Moro, S., Cortez, P. ve Rita, P. (2014). *A Data-Driven Approach to Predict the Success of Bank Telemarketing.* Decision Support Systems, 62, 22-31.

Veri, bir Portekiz bankasının 2008-2010 yılları arasındaki telefonla pazarlama kampanyalarından geliyor. Kişisel bilgi içermiyor; her satır anonim bir müşteriyle yapılan bir görüşme.

Hücreleri sırayla **Shift + Enter** ile çalıştır.

## 1. Veriyi indirmek

UCI dosyayı iç içe iki zip olarak veriyor: büyük zip'in içinde `bank-additional.zip`, onun içinde de CSV var. Dosyayı diske kaydetmeden, bellekte açıyoruz. Dosya bir kez indikten sonra aynı oturumda tekrar indirilmez.

In [ ]:
import io
import os
import zipfile

import numpy as np
import pandas as pd
import requests

DOSYA = "bank-additional-full.csv"
URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

def veriyi_indir():
    dis = zipfile.ZipFile(io.BytesIO(requests.get(URL, timeout=60).content))
    ic_adi = [a for a in dis.namelist() if a.endswith("bank-additional.zip")][0]
    ic = zipfile.ZipFile(io.BytesIO(dis.read(ic_adi)))
    csv_adi = [a for a in ic.namelist() if a.endswith(DOSYA)][0]
    with open(DOSYA, "wb") as f:
        f.write(ic.read(csv_adi))

if not os.path.exists(DOSYA):
    veriyi_indir()

df = pd.read_csv(DOSYA, sep=";")
print(df.shape)
df.head()

## 2. İlk bakış

Ayırıcının virgül değil noktalı virgül (`;`) olduğuna dikkat et. `sep` parametresini vermezsen pandas her satırı tek bir sütun sanar.

In [ ]:
df.info()

In [ ]:
print("Tekrarlanan satır:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

df["hedef"] = (df["y"] == "yes").astype(int)
print("Satır sayısı:", len(df))
print(f"Vadeli mevduat açanların oranı: %{df['hedef'].mean() * 100:.1f}")

Her 100 görüşmeden yalnızca 11'i "evet" ile bitmiş. Buna **dengesiz sınıf** deniyor ve modeli değerlendirirken işimizi zorlaştıracak. Birazdan neden olduğunu göreceğiz.

## 3. Gizli eksikler: `unknown` ve `999`

`isna()` bu veride sıfır gösterir ama bu, eksik veri olmadığı anlamına gelmiyor. Eksikler iki kılıkta saklanıyor:

- Kategorik sütunlarda `"unknown"` yazısı
- `pdays` sütununda `999` (müşteri daha önce hiç aranmamış demek)

In [ ]:
print("NaN toplamı:", df.isna().sum().sum())
print()
print("'unknown' sayıları:")
print((df == "unknown").sum().loc[lambda s: s > 0])
print()
print(f"pdays = 999 olan satırlar: %{(df['pdays'] == 999).mean() * 100:.1f}")

`default` sütununda (kredi borcunu ödememiş mi?) neredeyse her beş kişiden biri `unknown`. Bunları silmek yerine ayrı bir kategori olarak bırakıyoruz: "cevap vermemiş olmak" da bir bilgi.

## 4. Keşif: hangi grupta evet oranı yüksek?

Aynı işi birçok sütun için yapacağımızdan küçük bir fonksiyon yazıyoruz.

In [ ]:
def oran_tablosu(sutun, sira=None):
    t = df.groupby(sutun, observed=True)["hedef"].agg(gorusme="size", evet_orani="mean")
    if sira is not None:
        t = t.reindex(sira)
    t["evet_orani"] = (t["evet_orani"] * 100).round(1)
    return t

oran_tablosu("poutcome")

Önceki kampanyada "evet" demiş müşterilerin üçte ikisi bu kampanyada da evet demiş. Daha önce hiç aranmamış olanlarda oran %9'a düşüyor.

In [ ]:
AYLAR = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
oran_tablosu("month", AYLAR)

Mart, eylül, ekim ve aralıkta oran %40'ın üstünde. Ama bu aylarda yapılan görüşme sayısı çok az. Mayıs'ta 13 binden fazla görüşme var ve oran %6,4. Bu tablo "mart ayında arayın" demiyor; az aranan aylarda bankanın belki de zaten ilgili müşterileri aradığını düşündürüyor. **Korelasyon, nedensellik değildir.**

In [ ]:
oran_tablosu("contact")

In [ ]:
df["yas_grubu"] = pd.cut(df["age"], bins=[16, 25, 35, 45, 55, 65, 100],
                         labels=["17-25", "26-35", "36-45", "46-55", "56-65", "66+"])
oran_tablosu("yas_grubu")

In [ ]:
df["arama_sayisi"] = pd.cut(df["campaign"], bins=[0, 1, 2, 3, 5, 100],
                            labels=["1", "2", "3", "4-5", "6+"])
oran_tablosu("arama_sayisi")

Aynı müşteriyi ne kadar çok ararsan evet oranı o kadar düşüyor. Altı ve daha fazla aranmış kişilerde oran %5,5.

## 5. `duration` tuzağı

`duration`, görüşmenin saniye cinsinden süresi. Evet diyenlerle konuşma doğal olarak uzun sürüyor; bu yüzden bu sütun sonucu çok iyi tahmin ediyor gibi görünür. Ama arama yapılmadan önce görüşmenin ne kadar süreceğini bilemezsin. Veri setinin kendi açıklaması da gerçekçi bir model için bu sütunun çıkarılmasını öneriyor.

Gelecekte elinde olmayacak bir bilgiyi modele vermeye **veri sızıntısı** (data leakage) deniyor.

In [ ]:
df.groupby("y")["duration"].median()

## 6. Model: lojistik regresyon

Amaç: arama yapmadan önce, hangi müşterinin evet deme ihtimali daha yüksek, bunu sıralamak.

- `duration` ve keşif için eklediğimiz yardımcı sütunları çıkarıyoruz.
- Kategorik sütunları `OneHotEncoder` ile 0-1 sütunlarına, sayısal sütunları `StandardScaler` ile aynı ölçeğe getiriyoruz.
- `class_weight="balanced"`: az olan "evet" sınıfına daha fazla ağırlık ver demek.
- `stratify=y`: eğitim ve test setinde evet oranı aynı kalsın.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CIKAR = ["y", "hedef", "duration", "yas_grubu", "arama_sayisi"]
X = df.drop(columns=CIKAR)
y = df["hedef"]

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

kategorik = X.select_dtypes(exclude="number").columns.tolist()
sayisal = X.select_dtypes(include="number").columns.tolist()

def model_kur(kategorik, sayisal):
    hazirlik = ColumnTransformer([
        ("kategorik", OneHotEncoder(handle_unknown="ignore"), kategorik),
        ("sayisal", StandardScaler(), sayisal),
    ])
    return Pipeline([
        ("hazirlik", hazirlik),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])

model = model_kur(kategorik, sayisal)
model.fit(X_egitim, y_egitim)
olasilik = model.predict_proba(X_test)[:, 1]
tahmin = (olasilik >= 0.5).astype(int)
print("Eğitim:", len(X_egitim), "satır | Test:", len(X_test), "satır")

## 7. Accuracy neden yanıltır?

Herkese "hayır" diyen, hiçbir şey öğrenmemiş bir model düşün. Doğruluğu (accuracy) ne olurdu?

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

herkese_hayir = np.zeros(len(y_test), dtype=int)
print(f"Herkese 'hayır' diyen model accuracy: %{accuracy_score(y_test, herkese_hayir) * 100:.1f}")
print(f"Bizim modelin accuracy'si:           %{accuracy_score(y_test, tahmin) * 100:.1f}")

Hiçbir şey öğrenmemiş model daha "başarılı" görünüyor. Oysa bankaya tek bir müşteri bulamaz. Bu yüzden dengesiz veride başka metriklere bakıyoruz:

- **Recall (duyarlılık):** Gerçekten evet diyenlerin yüzde kaçını yakaladık?
- **Precision (kesinlik):** Evet dediğimiz kişilerin yüzde kaçı gerçekten evet dedi?
- **ROC-AUC:** Model, rastgele seçilmiş bir "evet"i rastgele seçilmiş bir "hayır"ın önüne ne sıklıkla koyuyor? 0,5 yazı tura demek, 1 kusursuz demek.

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, tahmin).ravel()
print(f"Doğru yakalanan evet: {tp} | Kaçırılan evet: {fn}")
print(f"Boşa aranan (hayır dediği halde evet tahmin edilen): {fp} | Doğru hayır: {tn}")
print()
print(classification_report(y_test, tahmin, target_names=["hayır", "evet"], digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, olasilik):.3f}")

## 8. Asıl soru: kimi önce aramalı?

Banka herkesi arayamaz. Modelin verdiği olasılığa göre müşterileri sıralayıp onluk dilimlere ayıralım: 1. dilim, modelin en umutlu bulduğu %10.

In [ ]:
sirali = pd.DataFrame({"gercek": y_test.to_numpy(), "olasilik": olasilik})
sirali = sirali.sort_values("olasilik", ascending=False).reset_index(drop=True)
sirali["dilim"] = sirali.index * 10 // len(sirali) + 1

dilimler = sirali.groupby("dilim")["gercek"].agg(kisi="size", evet="sum", evet_orani="mean")
dilimler["evet_orani"] = (dilimler["evet_orani"] * 100).round(1)

ortalama = y_test.mean() * 100
ilk = sirali[sirali["dilim"] == 1]
print(f"Rastgele arasan evet oranı:  %{ortalama:.1f}")
print(f"Modelin ilk %10'unda:        %{ilk['gercek'].mean() * 100:.1f}")
print(f"İlk %10, tüm evetlerin %{ilk['gercek'].sum() / sirali['gercek'].sum() * 100:.0f}'ini yakalıyor")
dilimler

## 9. Grafik

In [ ]:
import matplotlib.pyplot as plt

BOGURTLEN, MURDUM, SOLUK, GRI = "#C2185B", "#2B1036", "#D9CCDD", "#6B6475"

def yuzde(x):
    return f"%{x:.0f}"

fig, ax = plt.subplots(figsize=(10, 5.6), facecolor="white")
fig.subplots_adjust(top=0.78, bottom=0.14, left=0.05, right=0.97)

renkler = [BOGURTLEN] + [SOLUK] * 9
ax.bar(dilimler.index, dilimler["evet_orani"], color=renkler, width=0.82)
for d, oran in dilimler["evet_orani"].items():
    ax.text(d, oran + 1, yuzde(oran), ha="center", fontsize=11,
            fontweight="bold" if d == 1 else "normal", color=MURDUM, zorder=3,
            bbox=dict(facecolor="white", edgecolor="none", pad=1))

ax.axhline(ortalama, color=MURDUM, linestyle="--", linewidth=1.2, zorder=1)
ax.text(10.45, ortalama + 1.2, f"Rastgele arama: {yuzde(ortalama)}", ha="right", fontsize=10, color=MURDUM)

ax.set_xticks(dilimler.index)
ax.set_xticklabels([f"{d}." for d in dilimler.index], fontsize=11, color=MURDUM)
ax.set_xlabel("Modelin sıralamasına göre müşteri dilimi (her biri %10)", fontsize=10.5, color=GRI)
ax.set_yticks([])
for k in ["top", "right", "left"]:
    ax.spines[k].set_visible(False)
ax.spines["bottom"].set_color(SOLUK)
ax.tick_params(axis="x", length=0)
ax.set_ylim(0, dilimler["evet_orani"].max() * 1.15)

fig.text(0.05, 0.92, "Modelin seçtiği ilk %10'da neredeyse her iki kişiden biri evet dedi",
         fontsize=16, fontweight="bold", color=MURDUM)
fig.text(0.05, 0.86, "Test setindeki müşteriler, vadeli mevduat açma olasılığına göre sıralandı",
         fontsize=11, color=GRI)
fig.text(0.97, 0.02, "Kaynak: UCI Bank Marketing (CC BY 4.0) · haticeozbolat.com",
         ha="right", fontsize=8.5, color=GRI)
fig.savefig("banka-dilim-grafigi.png", dpi=200)
plt.show()

## 10. Sızıntılı model ne kadar "iyi" görünürdü?

Aynı modeli bir de `duration` sütunu dahil kuralım.

In [ ]:
X_sizinti = df.drop(columns=["y", "hedef", "yas_grubu", "arama_sayisi"])
Xs_egitim, Xs_test, ys_egitim, ys_test = train_test_split(
    X_sizinti, y, test_size=0.25, stratify=y, random_state=42)

sayisal_s = X_sizinti.select_dtypes(include="number").columns.tolist()
sizintili = model_kur(kategorik, sayisal_s).fit(Xs_egitim, ys_egitim)

print(f"duration olmadan ROC-AUC: {roc_auc_score(y_test, olasilik):.3f}")
print(f"duration ile ROC-AUC:     {roc_auc_score(ys_test, sizintili.predict_proba(Xs_test)[:, 1]):.3f}")

İkinci skor çok daha parlak ama gerçek hayatta kullanılamaz: arama bitmeden süresini bilemezsin. Bir model "fazla iyi" çıktığında ilk soracağın soru şu olmalı: **Bu bilgi, tahmin anında elimde olacak mı?**

## Kendin dene

1. `class_weight="balanced"` parametresini silip modeli yeniden kur. Recall ve precision nasıl değişti?
2. Eşiği 0,5 yerine 0,3 ve 0,7 yap (`olasilik >= 0.3`). Yakalanan ve boşa aranan kişi sayısı nasıl değişiyor?
3. `LogisticRegression` yerine `sklearn.ensemble.RandomForestClassifier` dene. ROC-AUC artıyor mu?
4. `education` sütunu için `oran_tablosu` çalıştır. Ne görüyorsun?

## Kaynak

Moro, S., Cortez, P. ve Rita, P. (2014). A Data-Driven Approach to Predict the Success of Bank Telemarketing. *Decision Support Systems*, 62, 22-31. Veri: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), CC BY 4.0.